In [ ]:
# --- imports ---
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# SETTINGS - edit these values as needed
# Keep the anchor symbol last in the list.
# ============================================================
sorted_symbols_list = ["SCHH", "USRT"]
moving_avg_days = 20
input_directory = Path("with_enhanced_prices")
output_directory = Path("with_unit_prices")

filename = f"{'_'.join(sorted_symbols_list)}.csv"
input_path = input_directory / filename
df = pd.read_csv(input_path, index_col=0)

anchor = sorted_symbols_list[-1]
for symbol in sorted_symbols_list:
    price_ratio_column = f"to {anchor} / to {symbol}"
    moving_average_column = f"{anchor} / {symbol} moving avg"
    prior_average_column = f"yday {moving_average_column}"

    df[moving_average_column] = (
        df[price_ratio_column].rolling(moving_avg_days).mean()
    )
    df[prior_average_column] = df[moving_average_column].shift(1)
    df[f"to {symbol} unit price"] = (
        df[prior_average_column] * df[f"to {symbol} price"]
    )

for symbol in sorted_symbols_list:
    df[f"{symbol} unit price pct diff"] = np.log(
        df[f"to {symbol} unit price"] / df[f"to {anchor} unit price"]
    )

non_anchor = sorted_symbols_list[0]
df["unit price pct diff"] = (
    df[f"{non_anchor} unit price pct diff"]
    - df[f"{anchor} unit price pct diff"]
)

output_directory.mkdir(parents=True, exist_ok=True)
output_path = output_directory / filename.replace(
    ".csv", f"_{moving_avg_days}.csv"
)
df.to_csv(output_path)

print(f"Saved {output_path}")
print("finished")
